# PROJECT SENTINEL — Notebook 2: Strategy Engine, Live Inference & Scheduler
**Layer C** (Probabilistic ML) + **Execution Layer** + Telegram + APScheduler

Sections:
- A: Model Training (adaptive labeling, XGBoost, validity gates)
- B: Live Inference (signal generation, position sizing, Telegram)
- C: Scheduler (hourly pipeline, retraining, 7am report)
- D: Report View (on-demand status dashboard)

## 0 — Setup

In [ ]:
import os, sys, json, logging
from datetime import datetime, timezone, timedelta

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

import config
from utils.data_utils   import update_all_tickers, load_base_csv
from utils.features     import build_all_features, compute_anchor_features, compute_macro_risk_state
from utils.labeling     import generate_labels, find_optimal_label_params
from utils.model_utils  import (train_xgboost, evaluate_model, check_validity,
                                 get_shap_drivers, save_model, load_model, list_valid_models)
from utils.signal_manager import (is_new_signal, register_signal, check_signal_aging,
                                   archive_signal, check_open_signals_status,
                                   get_active_signals, get_archived_signals,
                                   recalc_open_signal_qty)
from utils.telegram_utils import send_signal, send_morning_report, send_error_alert

os.makedirs(config.LOGS_DIR, exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    handlers=[
        logging.FileHandler(os.path.join(config.LOGS_DIR, 'scheduler.log')),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)
print('Setup complete.')

## Section A — Model Training

In [ ]:
# ── Load feature matrices ────────────────────────────────────────────────────
feature_dfs = {}
for ticker in config.ALTCOIN_TICKERS:
    safe = ticker.replace('/', '_')
    path = os.path.join(config.DATA_WORKING, f'{safe}_features.parquet')
    if os.path.exists(path):
        feature_dfs[ticker] = pd.read_parquet(path)
        print(f'Loaded {ticker}: {feature_dfs[ticker].shape}')
    else:
        print(f'WARNING: {path} not found. Run notebook 01 first.')

In [ ]:
# ── Define time-based splits ─────────────────────────────────────────────────
now   = pd.Timestamp.utcnow().tz_localize(None)
train_end  = now - pd.Timedelta(days=config.TRAIN_END_DAYS)
test1_start = now - pd.Timedelta(days=config.TEST1[0])
test1_end   = now - pd.Timedelta(days=config.TEST1[1])
test2_start = now - pd.Timedelta(days=config.TEST2[0])
test2_end   = now - pd.Timedelta(days=config.TEST2[1])

print(f'Training  : up to {train_end.date()}')
print(f'Test1     : {test1_start.date()} → {test1_end.date()}')
print(f'Test2 (OOS): {test2_start.date()} → {test2_end.date()}')

In [ ]:
def get_feature_cols(feat_df: pd.DataFrame) -> list[str]:
    """Return numeric feature columns (exclude timestamp and label if present)."""
    exclude = {'timestamp', 'label', 'open', 'high', 'low', 'close', 'volume'}
    cols = [c for c in feat_df.columns
            if c not in exclude and pd.api.types.is_numeric_dtype(feat_df[c])]
    return cols


def split_df(feat_df: pd.DataFrame):
    """Split feature DF into train / test1 / test2 using timestamp."""
    ts = pd.to_datetime(feat_df['timestamp'])
    train = feat_df[ts <= train_end].copy()
    t1    = feat_df[(ts > test1_start) & (ts <= test1_end)].copy()
    t2    = feat_df[(ts > test2_start) & (ts <= test2_end)].copy()
    return train, t1, t2

print('Helper functions defined.')

In [ ]:
# ── Train models for all altcoins ────────────────────────────────────────────
# (This cell may take several minutes due to the label grid search)

training_summary = []

for ticker in config.ALTCOIN_TICKERS:
    if ticker not in feature_dfs:
        print(f'[{ticker}] Skipped (no feature data).')
        continue

    feat = feature_dfs[ticker].copy()
    feat_cols = get_feature_cols(feat)
    train_df, test1_df, test2_df = split_df(feat)

    if len(train_df) < 200:
        print(f'[{ticker}] Insufficient training data ({len(train_df)} rows). Skipping.')
        continue

    print(f'\n[{ticker}] Grid searching label params...')
    params = find_optimal_label_params(train_df, feat_cols, direction='long', verbose=False)
    tp_pct, sl_pct = params['tp_pct'], params['sl_pct']
    k1, k2         = params['k1'],     params['k2']
    print(f'  Best params: TP={tp_pct:.3f} SL={sl_pct:.3f} k1={k1} k2={k2} '
          f'(score={params["best_score"]:.3f})')

    # Generate labels for all splits
    def add_labels(df):
        lbl = generate_labels(df, tp_pct, sl_pct, k1, k2)
        df  = df.copy()
        df['label'] = lbl.values
        return df.dropna(subset=['label'] + feat_cols[:3])  # drop NaN label rows

    train_l = add_labels(train_df)
    test1_l = add_labels(test1_df)
    test2_l = add_labels(test2_df)

    X_train = train_l[feat_cols]
    y_train = train_l['label'].astype(int)

    # Train model
    print(f'  Training XGBoost on {len(X_train)} samples...')
    model = train_xgboost(X_train, y_train)

    # Evaluate
    m1 = evaluate_model(model, test1_l[feat_cols], test1_l['label'].astype(int))
    m2 = evaluate_model(model, test2_l[feat_cols], test2_l['label'].astype(int))
    valid = check_validity(m1, m2)

    print(f'  Test1: PF={m1["PF"]:.3f} trades={m1["trade_count"]} wr={m1["win_rate"]:.2f}')
    print(f'  Test2: PF={m2["PF"]:.3f} trades={m2["trade_count"]} wr={m2["win_rate"]:.2f}')
    print(f'  Valid: {"✅" if valid else "❌"}')

    if valid:
        meta = {
            'ticker':          ticker,
            'feature_columns': feat_cols,
            'label_params':    {'tp_pct': tp_pct, 'sl_pct': sl_pct, 'k1': k1, 'k2': k2},
            'test1':           m1,
            'test2':           m2,
            'trained_at':      datetime.now(timezone.utc).isoformat(),
        }
        save_model(model, ticker, meta)

    training_summary.append({
        'ticker':       ticker,
        'tp_pct':       tp_pct,
        'sl_pct':       sl_pct,
        'PF_test1':     m1['PF'],
        'trades_test1': m1['trade_count'],
        'PF_test2':     m2['PF'],
        'trades_test2': m2['trade_count'],
        'valid':        '✅' if valid else '❌',
    })

print('\n--- Training complete ---')

In [ ]:
# ── Training summary table ───────────────────────────────────────────────────
summary_df = pd.DataFrame(training_summary).set_index('ticker')
summary_df.style.applymap(lambda v: 'color: green' if v == '✅' else ('color: red' if v == '❌' else ''),
                          subset=['valid'])

In [ ]:
# ── SHAP feature importance for valid models ─────────────────────────────────
valid_tickers = [r['ticker'] for r in training_summary if r['valid'] == '✅']
print(f'Valid models: {valid_tickers}')

for ticker in valid_tickers:
    model, meta = load_model(ticker)
    feat  = feature_dfs[ticker]
    train_df, _, _ = split_df(feat)
    feat_cols = meta['feature_columns']
    X_sample = train_df[feat_cols].dropna().tail(500)

    try:
        explainer  = shap.TreeExplainer(model)
        shap_vals  = explainer.shap_values(X_sample)
        mean_abs   = np.abs(shap_vals).mean(axis=0)
        top_n = 20
        top_idx = np.argsort(mean_abs)[::-1][:top_n]

        fig, ax = plt.subplots(figsize=(8, 6))
        ax.barh([feat_cols[i] for i in top_idx[::-1]], mean_abs[top_idx[::-1]], color='steelblue')
        ax.set_title(f'{ticker} — SHAP Feature Importance (top {top_n})')
        ax.set_xlabel('Mean |SHAP value|')
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f'[{ticker}] SHAP plot failed: {e}')

## Section B — Live Inference

In [ ]:
def run_live_inference(
    crypto_dfs: dict,
    macro_dfs: dict,
    send_telegram: bool = True,
) -> list[dict]:
    """
    Run live inference for all valid models.
    Returns list of triggered signals.
    """
    from utils.features import compute_anchor_features, compute_macro_risk_state
    from utils.features import compute_altcoin_features

    macro_state = compute_macro_risk_state(macro_dfs)
    btc_anchor  = compute_anchor_features(crypto_dfs['BTC/USDT'], 'BTC')
    eth_anchor  = compute_anchor_features(crypto_dfs['ETH/USDT'], 'ETH')

    valid_tickers = list_valid_models()
    signals_fired = []

    # Check TP/SL hits on existing open signals
    current_prices = {
        t: float(crypto_dfs[t]['close'].iloc[-1])
        for t in config.ALTCOIN_TICKERS if t in crypto_dfs
    }
    check_open_signals_status(current_prices)

    for ticker in valid_tickers:
        if ticker not in crypto_dfs:
            continue

        model, meta = load_model(ticker)
        if model is None:
            continue

        feat_cols = meta['feature_columns']
        lp        = meta['label_params']   # tp_pct, sl_pct, k1, k2

        # Build current feature vector
        df_coin  = crypto_dfs[ticker]
        feat_all = compute_altcoin_features(df_coin, btc_anchor, eth_anchor, macro_state)

        # Use last complete row
        feat_live = feat_all.dropna(subset=feat_cols).tail(1)
        if feat_live.empty:
            logger.warning(f'[{ticker}] No valid feature row for inference.')
            continue

        # Feature parity check
        try:
            X_live = feat_live[feat_cols]
        except KeyError as e:
            logger.error(f'[{ticker}] Feature parity violation: {e}')
            continue

        # Predict
        p_win  = float(model.predict_proba(X_live)[0, 1])
        close  = float(df_coin['close'].iloc[-1])
        atr    = float(feat_live['ATR_14'].iloc[0]) if 'ATR_14' in feat_live.columns else 0.0
        atr_n  = atr / close if close > 0 else 0.0

        drivers = get_shap_drivers(model, X_live, n=5)

        # ── LONG signal ───────────────────────────────────────────────────────
        if p_win >= config.LONG_THRESHOLD:
            tp  = close * (1 + lp['tp_pct'] + lp['k1'] * atr_n)
            sl  = close * (1 - lp['sl_pct'] - lp['k2'] * atr_n)
            qty = config.MAX_LOSS_USDT / max(abs(close - sl), 1e-8)

            if is_new_signal(ticker, 'LONG'):
                register_signal(ticker, 'LONG', close, tp, sl, qty, p_win,
                                lp['tp_pct'], lp['sl_pct'])
                if send_telegram:
                    send_signal(ticker, 'LONG', close, tp, sl, qty, p_win, drivers)
                signals_fired.append({'coin': ticker, 'direction': 'LONG', 'p_win': p_win,
                                      'entry': close, 'tp': tp, 'sl': sl, 'qty': qty})
                logger.info(f'[{ticker}] LONG signal fired P={p_win:.2f}')
            else:
                aging = check_signal_aging(ticker, close)
                if aging == 'repeat':
                    if send_telegram:
                        existing = [s for s in get_active_signals() if s['coin'] == ticker]
                        if existing:
                            s = existing[0]
                            send_signal(ticker, 'LONG', s['entry'], s['tp'], s['sl'],
                                        s['qty'], p_win, drivers, is_repeat=True)
                    logger.info(f'[{ticker}] LONG repeat signal sent.')

        # ── SHORT signal ──────────────────────────────────────────────────────
        elif p_win <= (1 - config.SHORT_THRESHOLD):
            tp  = close * (1 - lp['tp_pct'] - lp['k1'] * atr_n)
            sl  = close * (1 + lp['sl_pct'] + lp['k2'] * atr_n)
            qty = config.MAX_LOSS_USDT / max(abs(sl - close), 1e-8)

            if is_new_signal(ticker, 'SHORT'):
                register_signal(ticker, 'SHORT', close, tp, sl, qty, p_win,
                                lp['tp_pct'], lp['sl_pct'])
                if send_telegram:
                    send_signal(ticker, 'SHORT', close, tp, sl, qty, p_win, drivers)
                signals_fired.append({'coin': ticker, 'direction': 'SHORT', 'p_win': p_win,
                                      'entry': close, 'tp': tp, 'sl': sl, 'qty': qty})
                logger.info(f'[{ticker}] SHORT signal fired P={1-p_win:.2f}')
            else:
                aging = check_signal_aging(ticker, close)
                if aging == 'repeat':
                    if send_telegram:
                        existing = [s for s in get_active_signals() if s['coin'] == ticker]
                        if existing:
                            s = existing[0]
                            send_signal(ticker, 'SHORT', s['entry'], s['tp'], s['sl'],
                                        s['qty'], p_win, drivers, is_repeat=True)

    return signals_fired

In [ ]:
# ── Run live inference manually ───────────────────────────────────────────────
# Set send_telegram=True to actually fire Telegram messages
print('Running live inference...')
all_dfs     = update_all_tickers()
crypto_dfs  = {k: v for k, v in all_dfs.items() if '/' in k}
macro_dfs   = {k: v for k, v in all_dfs.items() if '/' not in k}

signals = run_live_inference(crypto_dfs, macro_dfs, send_telegram=False)

if signals:
    print(f'\n{len(signals)} signal(s) fired:')
    for s in signals:
        icon = '🟢' if s['direction'] == 'LONG' else '🔴'
        print(f"  {icon} {s['coin']} {s['direction']}  entry={s['entry']:.4f}  "
              f"TP={s['tp']:.4f}  SL={s['sl']:.4f}  qty={s['qty']:.4f}  P={s['p_win']:.2f}")
else:
    print('No signals triggered at this time.')

## Section C — Scheduler

In [ ]:
from apscheduler.schedulers.background import BackgroundScheduler
from apscheduler.events import EVENT_JOB_ERROR

# ── Scheduled job functions ──────────────────────────────────────────────────

def run_hourly_pipeline():
    """Fetch latest data → rebuild features → run inference → update signal states."""
    logger.info('=== Hourly pipeline START ===')
    try:
        dfs       = update_all_tickers()
        c_dfs     = {k: v for k, v in dfs.items() if '/' in k}
        m_dfs     = {k: v for k, v in dfs.items() if '/' not in k}
        if not c_dfs.get('BTC/USDT') is None:
            run_live_inference(c_dfs, m_dfs, send_telegram=True)
        else:
            logger.error('BTC/USDT data missing — skipping inference.')
    except Exception as e:
        logger.error(f'Hourly pipeline error: {e}')
        send_error_alert(f'Hourly pipeline error: {e}')
    logger.info('=== Hourly pipeline END ===')


def run_retrain():
    """Re-run full training for all coins and replace models if they pass validity."""
    logger.info('=== Retraining START ===')
    try:
        dfs      = update_all_tickers()
        c_dfs    = {k: v for k, v in dfs.items() if '/' in k}
        m_dfs    = {k: v for k, v in dfs.items() if '/' not in k}
        feat_dfs = build_all_features(c_dfs, m_dfs)

        # Save updated features
        for ticker, feat in feat_dfs.items():
            safe = ticker.replace('/', '_')
            feat.to_parquet(os.path.join(config.DATA_WORKING, f'{safe}_features.parquet'), index=False)

        # Retrain loop (same as Section A)
        now_r     = pd.Timestamp.utcnow().tz_localize(None)
        train_end_r = now_r - pd.Timedelta(days=config.TRAIN_END_DAYS)
        t1s = now_r - pd.Timedelta(days=config.TEST1[0])
        t1e = now_r - pd.Timedelta(days=config.TEST1[1])
        t2s = now_r - pd.Timedelta(days=config.TEST2[0])
        t2e = now_r - pd.Timedelta(days=config.TEST2[1])

        for ticker in config.ALTCOIN_TICKERS:
            if ticker not in feat_dfs:
                continue
            feat = feat_dfs[ticker]
            feat_cols = get_feature_cols(feat)
            ts = pd.to_datetime(feat['timestamp'])
            train_df = feat[ts <= train_end_r]
            test1_df = feat[(ts > t1s) & (ts <= t1e)]
            test2_df = feat[(ts > t2s) & (ts <= t2e)]

            if len(train_df) < 200:
                continue

            params = find_optimal_label_params(train_df, feat_cols, verbose=False)
            tp_pct, sl_pct = params['tp_pct'], params['sl_pct']
            k1, k2         = params['k1'],     params['k2']

            def add_lbl(df):
                lbl = generate_labels(df, tp_pct, sl_pct, k1, k2)
                df  = df.copy()
                df['label'] = lbl.values
                return df.dropna(subset=['label'])

            train_l = add_lbl(train_df)
            test1_l = add_lbl(test1_df)
            test2_l = add_lbl(test2_df)

            model = train_xgboost(train_l[feat_cols], train_l['label'].astype(int))
            m1 = evaluate_model(model, test1_l[feat_cols], test1_l['label'].astype(int))
            m2 = evaluate_model(model, test2_l[feat_cols], test2_l['label'].astype(int))

            if check_validity(m1, m2):
                meta = {
                    'ticker': ticker, 'feature_columns': feat_cols,
                    'label_params': {'tp_pct': tp_pct, 'sl_pct': sl_pct, 'k1': k1, 'k2': k2},
                    'test1': m1, 'test2': m2,
                    'trained_at': datetime.now(timezone.utc).isoformat(),
                }
                save_model(model, ticker, meta)
                logger.info(f'[{ticker}] Model retrained and saved.')
            else:
                logger.info(f'[{ticker}] Retrain failed validity — old model kept.')
    except Exception as e:
        logger.error(f'Retrain error: {e}')
        send_error_alert(f'Retrain error: {e}')
    logger.info('=== Retraining END ===')


def send_morning_report_job():
    """Build and send the 7am daily Telegram report."""
    logger.info('Sending morning report...')
    try:
        dfs = update_all_tickers()
        current_prices = {
            t: float(dfs[t]['close'].iloc[-1])
            for t in config.ALTCOIN_TICKERS if t in dfs and not dfs[t].empty
        }
        since_24h = datetime.now(timezone.utc) - timedelta(hours=24)
        active   = get_active_signals()
        archived = get_archived_signals(since=since_24h)
        send_morning_report(active, archived, current_prices)
    except Exception as e:
        logger.error(f'Morning report error: {e}')


def on_scheduler_error(event):
    logger.error(f'Scheduler job failed: {event.exception}')

print('Scheduler job functions defined.')

In [ ]:
# ── Start the scheduler ───────────────────────────────────────────────────────
# Runs in background while this cell (and the keep-alive below) execute.
# To stop: call scheduler.shutdown()

scheduler = BackgroundScheduler(timezone='UTC')
scheduler.add_listener(on_scheduler_error, EVENT_JOB_ERROR)

# Hourly inference pipeline
scheduler.add_job(
    run_hourly_pipeline,
    'interval',
    minutes=config.SCHEDULER_INTERVAL_MIN,
    id='hourly_pipeline',
    misfire_grace_time=300,
    coalesce=True,
)

# Periodic retraining
scheduler.add_job(
    run_retrain,
    'interval',
    hours=config.RETRAIN_INTERVAL_HOURS,
    id='retrain',
    misfire_grace_time=1800,
    coalesce=True,
)

# Daily 7am report
scheduler.add_job(
    send_morning_report_job,
    'cron',
    hour=config.MORNING_REPORT_HOUR,
    minute=0,
    id='morning_report',
)

scheduler.start()
print(f'Scheduler started. Next inference run in ~{config.SCHEDULER_INTERVAL_MIN} minutes.')
print('Keep this notebook running. To stop: scheduler.shutdown()')

In [ ]:
# ── Keep-alive (blocks notebook, runs scheduler) ──────────────────────────────
# Stop this cell to shut down the scheduler gracefully.
import time
try:
    while True:
        time.sleep(30)
except KeyboardInterrupt:
    scheduler.shutdown()
    print('Scheduler stopped.')

## Section D — Report View (On-Demand)
Run these cells at any time to see current system status.

In [ ]:
# ── Active signals ────────────────────────────────────────────────────────────
dfs_now = update_all_tickers()
current_prices = {
    t: float(dfs_now[t]['close'].iloc[-1])
    for t in config.ALTCOIN_TICKERS if t in dfs_now and not dfs_now[t].empty
}

active = get_active_signals()
if active:
    rows = []
    for s in active:
        cp  = current_prices.get(s['coin'], s['entry'])
        rr  = recalc_open_signal_qty(s, cp)
        rows.append({
            'Coin':       s['coin'],
            'Direction':  s['direction'],
            'Entry':      s['entry'],
            'TP':         s['tp'],
            'SL':         s['sl'],
            'Qty':        s['qty'],
            'P(win)':     s['p_win'],
            'Current':    round(cp, 4),
            'Repeats':    s['repeat_count'],
            'New RR':     round(rr['rr'], 2) if rr else '< 1:1',
            'New Qty':    round(rr['qty'], 4) if rr else '—',
        })
    pd.DataFrame(rows).set_index('Coin')
else:
    print('No active signals.')

In [ ]:
# ── Model validity summary ────────────────────────────────────────────────────
import json
rows = []
for ticker in list_valid_models():
    _, meta = load_model(ticker)
    if meta:
        rows.append({
            'Ticker':       ticker,
            'PF_test1':     meta['test1']['PF'],
            'trades_test1': meta['test1']['trade_count'],
            'PF_test2':     meta['test2']['PF'],
            'trades_test2': meta['test2']['trade_count'],
            'trained_at':   meta.get('trained_at', '—')[:10],
        })
if rows:
    pd.DataFrame(rows).set_index('Ticker')
else:
    print('No saved models found. Run Section A first.')

In [ ]:
# ── Data freshness ────────────────────────────────────────────────────────────
freshness = []
for ticker in config.CRYPTO_TICKERS:
    df = load_base_csv(ticker, 'crypto')
    if not df.empty:
        last = pd.Timestamp(df['timestamp'].max())
        age  = pd.Timestamp.utcnow().tz_localize(None) - last
        freshness.append({'ticker': ticker, 'last_candle': str(last)[:16],
                          'age_hours': round(age.total_seconds()/3600, 1)})
for name in config.MACRO_TICKERS:
    df = load_base_csv(name, 'macro')
    if not df.empty:
        last = pd.Timestamp(df['timestamp'].max())
        age  = pd.Timestamp.utcnow().tz_localize(None) - last
        freshness.append({'ticker': name, 'last_candle': str(last)[:16],
                          'age_hours': round(age.total_seconds()/3600, 1)})
pd.DataFrame(freshness).set_index('ticker').sort_values('age_hours', ascending=False)

In [ ]:
# ── Last 24h archived signals ─────────────────────────────────────────────────
since = datetime.now(timezone.utc) - timedelta(hours=24)
archived = get_archived_signals(since=since)
if archived:
    rows = [{
        'Coin':    s['coin'],
        'Dir':     s['direction'],
        'Entry':   s['entry'],
        'Result':  s.get('archive_reason', '—'),
        'At':      s['first_signal_at'][:16],
    } for s in archived]
    pd.DataFrame(rows)
else:
    print('No archived signals in last 24h.')